# Bleed-through classification

Set `IMAGE_PATH` in the first code cell and select **Run All**. Use a Python 3 kernel with NumPy, OpenCV and Matplotlib. If needed, install them once with `%pip install numpy opencv-python matplotlib`.

The notebook reports whether a single scan shows signs of reverse-side ink and displays the candidate marks.

## Choose an image

The example uses a bundled H-DIBCO scan. Change the path to try another scan.

In [ ]:
IMAGE_PATH = "joeal_data/hdibco2014/images/H08.png"

## 1. Load the scan

OpenCV reads the scan as an 8-bit image. A relative path starts from the notebook kernel’s working folder. The original preview uses colour, and the algorithm uses grayscale.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

input_path = Path(IMAGE_PATH).expanduser()
if not input_path.is_absolute() and not input_path.is_file():
    input_path = Path.cwd().parent / input_path
input_path = input_path.resolve()
if not input_path.is_file():
    raise FileNotFoundError(f"Set IMAGE_PATH to an existing image file. Tried {input_path}")

original = cv2.imread(str(input_path), cv2.IMREAD_COLOR)
if original is None:
    raise ValueError(f"OpenCV could not decode this image: {input_path}")
if min(original.shape[:2]) < 8:
    raise ValueError("The scan needs to be at least 8 pixels wide and tall.")

## Detection helpers

### Luminance

Convert BGR colour to one brightness value per pixel. Green gets the largest weight, followed by red and blue. The integer arithmetic rounds the weighted result back to an 8-bit pixel. A grayscale input passes through directly.

In [ ]:
def luminance(image):
    """Convert an OpenCV-order colour array to 8-bit grayscale."""
    if image.ndim == 2:
        return image
    blue, green, red = np.moveaxis(image.astype(np.int32), -1, 0)
    return ((3735 * blue + 19235 * green + 9798 * red + 16384) >> 15).astype(np.uint8)

### Area resize

Keep detection affordable by shrinking large scans. Each output pixel averages the source area it covers, including partial pixels at its borders. Cumulative sums let the inner helper calculate those averages efficiently in each direction.

In [ ]:
def area_resize(image, height, width):
    """Average the source pixel areas covered by each output pixel."""
    def along_axis(values, size):
        length = values.shape[-1]
        if size == length:
            return values
        boundaries = np.linspace(0, length, size + 1)
        left = boundaries.astype(int)
        cumulative = np.concatenate(
            (np.zeros_like(values[..., :1]), np.cumsum(values, axis=-1)), axis=-1
        )
        integrals = cumulative[..., left]
        integrals += values[..., np.minimum(left, length - 1)] * (boundaries - left)
        return np.diff(integrals, axis=-1) * (size / length)

    rows = along_axis(image.astype(np.float64), width)
    return np.rint(along_axis(rows.T, height).T).clip(0, 255).astype(np.uint8)

### One-dimensional filtering

Slide a row of weights along the chosen axis and sum the weighted neighbours for each pixel. Reflecting the image at the border gives the filter values to use near the edge. Gaussian smoothing and Sobel gradients both use this helper.

In [ ]:
def convolve_axis(image, kernel, axis):
    """Apply a one-dimensional kernel with reflected borders."""
    radius = len(kernel) // 2
    padding = [(0, 0)] * image.ndim
    padding[axis] = (radius, radius)
    padded = np.moveaxis(np.pad(image, padding, mode="reflect"), axis, 0)
    result = np.zeros_like(padded[radius:radius + image.shape[axis]], dtype=np.float32)
    for offset, weight in enumerate(kernel):
        result += weight * padded[offset:offset + image.shape[axis]]
    return np.moveaxis(result, 0, axis)

### Gaussian smoothing

Generate weights that decay with distance from the centre, then normalise them to add up to one. Applying these weights horizontally and vertically produces a smooth estimate of the page lighting. Larger sigma values smooth over a wider area.

In [ ]:
def gaussian(image, sigma):
    radius = round(4 * sigma)
    offsets = np.arange(-radius, radius + 1, dtype=np.float64)
    weights = np.exp(-0.5 * (offsets / sigma) ** 2)
    weights /= weights.sum()
    horizontal = convolve_axis(image, weights.astype(np.float32), 1)
    return convolve_axis(horizontal, weights.astype(np.float32), 0)

### Sobel edge strength

Lightly smooth the image, then estimate the horizontal and vertical changes in brightness with Sobel filters. Combine the two directions into one edge-strength value per pixel. Strong writing usually changes brightness more sharply than faint ghost strokes.

In [ ]:
def sobel_magnitude(image):
    smooth = (0.25, 0.5, 0.25)
    blurred = convolve_axis(convolve_axis(image, smooth, 1), smooth, 0)
    blurred = np.rint(blurred)
    derivative = (-1, 0, 1)
    smoothing = (1, 2, 1)
    dx = convolve_axis(convolve_axis(blurred, derivative, 1), smoothing, 0)
    dy = convolve_axis(convolve_axis(blurred, derivative, 0), smoothing, 1)
    return np.hypot(dx, dy)

### Mask expansion

Grow each marked pixel by a square neighbourhood of the requested radius. The classifier uses it to exclude faint pixels close to dark ink and to find the pixels along the dark ink boundary.

In [ ]:
def expand_mask(mask, radius, outside=False):
    """Mark pixels covered by a square neighbourhood of a foreground pixel."""
    height, width = mask.shape
    padded = np.pad(mask, radius, constant_values=outside)
    expanded = np.zeros_like(mask)
    for row in range(2 * radius + 1):
        for column in range(2 * radius + 1):
            expanded |= padded[row:row + height, column:column + width]
    return expanded

### Connected regions

Scan each row for runs of marked pixels. Join runs in adjacent rows when they touch horizontally or diagonally, then give each group a label and record its bounding box and pixel count. The classifier uses those measurements to decide which marks are plausible candidates.

In [ ]:
def connected_regions(mask):
    """Label 8-connected foreground by joining touching row runs."""
    parent = [0]
    runs = []
    previous = []

    def root(label):
        while parent[label] != label:
            parent[label] = parent[parent[label]]
            label = parent[label]
        return label

    for y, row in enumerate(mask):
        boundaries = np.flatnonzero(np.diff(row, prepend=False, append=False))
        current = []
        cursor = 0
        for start, end in boundaries.reshape(-1, 2):
            label = len(parent)
            parent.append(label)
            current.append((start, end, label))
            runs.append((y, start, end, label))
            while cursor < len(previous) and previous[cursor][1] < start:
                cursor += 1
            for index in range(cursor, len(previous)):
                left, _, predecessor = previous[index]
                if left > end:
                    break
                parent[root(predecessor)] = label
        previous = current

    labels = np.zeros(mask.shape, dtype=np.int32)
    regions = {}
    for y, start, end, label in runs:
        component = root(label)
        labels[y, start:end] = component
        x0, y0, x1, y1, area = regions.get(component, (start, y, end, y + 1, 0))
        regions[component] = (min(x0, start), y0, max(x1, end), y + 1, area + end - start)
    return labels, regions

### Histogram counts and sums

Count how often each brightness value appears. Build running totals for both pixel count and brightness so Otsu can calculate the size and mean of any intensity range by subtracting two totals.

In [ ]:
def _groups(image):
    counts = np.bincount(image.ravel(), minlength=256)
    levels = np.flatnonzero(counts)
    counts = counts[levels]
    weight = np.r_[0, np.cumsum(counts, dtype=np.float64)]
    moment = np.r_[0, np.cumsum(counts * levels, dtype=np.float64)]
    return levels, weight, moment

### Three-class Otsu

Try pairs of cutoffs that split the histogram into three non-empty groups. Score each pair by how well it separates the groups’ average brightness, accounting for how many pixels each group contains. The detector uses the winning cutoffs to find strong ink and middle-tone candidates.

In [ ]:
def threshold_multiotsu(image, classes=3):
    """Find two histogram splits that maximise variation between three groups."""
    if classes != 3:
        raise ValueError("Only three-class Otsu is implemented.")
    levels, weight, moment = _groups(image)
    if len(levels) < 3:
        raise ValueError("Three-class Otsu needs at least three intensity levels.")

    # Maximising the sum of count times mean squared is equivalent to
    # maximising between-class variance: the overall mean is fixed.
    best_score = -1.0
    best_pair = (1, 2)
    end = len(levels)
    for first in range(1, end - 1):
        second = np.arange(first + 1, end)
        scores = (
            moment[first] ** 2 / weight[first]
            + (moment[second] - moment[first]) ** 2 / (weight[second] - weight[first])
            + (moment[end] - moment[second]) ** 2 / (weight[end] - weight[second])
        )
        candidate = int(np.argmax(scores))
        if scores[candidate] > best_score:
            best_score = scores[candidate]
            best_pair = (first, int(second[candidate]))
    return tuple(int(levels[split - 1]) for split in best_pair)

## Classify the original scan

`bleedthrough_features` scales the longest side down to at most 1,200 pixels, corrects uneven lighting and splits brightness into three ranges. It treats the darkest range as an ink reference and looks for groups of middle-tone pixels away from that ink. It filters the groups by size, shape, fill and border contact.

The **tone score** is the middle-tone fraction of the page. The **component score** divides the area of retained groups by the dark ink area. The **edge score** makes the same comparison after giving more weight to groups whose edges are weaker than the dark ink edges.

The notebook uses the **edge flag** as the decision, matching the original pipeline. It requires a score of at least 0.04 and enough dark ink and candidate marks for the comparison. This is a heuristic from the project experiment. Faint front-side writing and ghost writing can look similar, and some subtle bleed-through can be missed.

In [ ]:
# Adapted from joeal_notebooks/Bleedthrough_detection_experiment.ipynb (Joeal branch).
THRESHOLDS = dict(tone=0.01, components=0.08, edges=0.04)


def bleedthrough_features(image, max_side=1200):
    if image is None or not isinstance(image,np.ndarray) or image.size == 0:
        raise ValueError('Expected a non-empty image')
    if image.dtype != np.uint8: raise ValueError('Expected uint8 input in [0,255]')
    if image.ndim == 2: gray = image
    elif image.ndim == 3 and image.shape[2] == 3: gray = luminance(image)
    else: raise ValueError('Expected greyscale or BGR image')
    if min(gray.shape)<8: raise ValueError('Image must be at least 8 pixels in each dimension')
    scale = min(1., max_side/max(gray.shape))
    size = (max(8,round(gray.shape[1]*scale)),max(8,round(gray.shape[0]*scale)))
    small_gray = area_resize(gray, size[1], size[0])
    # Detection-only lighting correction: broad scale preserves character-size marks.
    sigma = max(3.,0.08*min(small_gray.shape))
    illumination = gaussian(small_gray.astype(np.float32), sigma)
    norm = np.clip(small_gray.astype(np.float32)/np.maximum(illumination,1)*230,0,255).astype(np.uint8)
    h,w = norm.shape
    # Constant scans have no credible weak-ink class.
    if np.unique(norm).size >= 3:
        low,high = threshold_multiotsu(norm,classes=3)
        middle = (norm>low)&(norm<high)
        fg = norm<=low
    else:
        low=high=0; middle=np.zeros_like(norm,dtype=bool);fg=np.zeros_like(middle)
    # A fixed analysis-scale separation prevents foreground stroke edges dominating.
    near_fg = expand_mask(fg, radius=2)
    labels, regions = connected_regions(middle & ~near_fg)
    gradient = sobel_magnitude(small_gray)
    # Compare stroke boundaries, not interiors with zero gradient.
    fg_boundary = expand_mask(fg, radius=1) & expand_mask(~fg, radius=1)
    fg_grad = float(np.percentile(gradient[fg_boundary],75)) if fg_boundary.any() else 0.
    kept=[]; weighted_area=0.; details=[]
    for i, (x, y, right, bottom, area) in regions.items():
        bw, bh = right - x, bottom - y
        extent=area/(bw*bh); aspect=bw/max(bh,1)
        touches=x==0 or y==0 or x+bw==w or y+bh==h
        if area<5 or area>0.01*h*w or not (0.08<aspect<15) or extent<0.08:
            continue
        if touches and (area/max(bw,bh)<3): continue
        region = np.s_[y:bottom, x:right]
        component = labels[region] == i
        comp_grad = float(np.percentile(gradient[region][component], 75))
        weakness=float(np.clip(1-comp_grad/max(fg_grad,1),0,1))
        kept.append(i); weighted_area+=area*weakness
        details.append(dict(component=i,area=int(area),gradient=comp_grad,weakness=weakness))
    candidates=np.isin(labels,kept)
    area=int(candidates.sum()); fg_area=int(fg.sum())
    credible=fg_area>=0.002*h*w and len(kept)>=3 and area>=0.001*h*w
    scores=dict(tone=float(middle.mean()),components=area/max(fg_area,1),edges=weighted_area/max(fg_area,1))
    flags=dict(tone=scores['tone']>=THRESHOLDS['tone'],
      components=bool(credible and scores['components']>=THRESHOLDS['components']),
      edges=bool(credible and scores['edges']>=THRESHOLDS['edges']))
    return dict(scores=scores,flags=flags,credible=bool(credible),num_components=len(kept),
      candidate_fraction=float(candidates.mean()),foreground_fraction=float(fg.mean()),
      thresholds=(int(low),int(high)),middle=middle,candidates=candidates,
      analysis_gray=small_gray,analysis_normalised=norm,dark_ink_reference=fg,components=details)

## Display the classification

Run the detector on the original scan, print its decision and scores, and show where it found candidate marks. The right panel shows the marks at the detector’s analysis resolution.

In [ ]:
features = bleedthrough_features(original)
detected = features["flags"]["edges"]
print(f"Bleed-through {'detected' if detected else 'not detected'}")
for name, score in features["scores"].items():
    print(f"{name.capitalize()} score: {score:.4f}")
print(f"Candidate groups: {features['num_components']}")

figure, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original scan")
axes[1].imshow(features["candidates"], cmap="gray_r", vmin=0, vmax=1)
axes[1].set_title("Candidate marks in black")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()
plt.close(figure)